In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import SegformerForSemanticSegmentation
import timm
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ValSegFusionDataset(Dataset):
    def __init__(self, image_dir, mask_dir, img_size=(512, 512)):
        self.mask_dir = mask_dir
        self.img_size = img_size
        self.mask_filenames = [f for f in os.listdir(mask_dir) if f.endswith('.png')]
        
        self.image_path_map = {}
        for path in Path(image_dir).rglob("*.jpg"):
            self.image_path_map[path.name] = str(path)
            
        self.img_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.mask_filenames)

    def __getitem__(self, idx):
        mask_name = self.mask_filenames[idx]
        img_name = mask_name.replace('.png', '.jpg')
        img_path = self.image_path_map.get(img_name)

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(os.path.join(self.mask_dir, mask_name)).convert("L") 

        image = image.resize(self.img_size, Image.BILINEAR)
        mask = mask.resize(self.img_size, Image.NEAREST)

        return self.img_transform(image), torch.tensor(np.array(mask), dtype=torch.long), img_path

val_image_dir =  r"D:\Study\HumanStudy\Dataset\Validation\01.원천데이터"
val_mask_dir =  r"D:\Study\HumanStudy\Dataset\Validation_Masks_2"

val_dataset = ValSegFusionDataset(val_image_dir, val_mask_dir)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0)

print(f"{len(val_dataset)}장")

52500장


In [2]:
classifier = timm.create_model('efficientnet_b0', pretrained=False, num_classes=11)
classifier.load_state_dict(torch.load('best_efficientnet_b0_multi_label.pth', weights_only=True))
classifier = classifier.to(device)
classifier.eval()

class FusionSegFormer(nn.Module):
    def __init__(self, num_classes=11):
        super().__init__()
        self.segformer = SegformerForSemanticSegmentation.from_pretrained(
            "nvidia/mit-b0", num_labels=num_classes, ignore_mismatched_sizes=True, use_safetensors=True
        )
        self.fusion_conv = nn.Conv2d(in_channels=22, out_channels=11, kernel_size=1)

    def forward(self, pixel_values, hints):
        seg_logits = self.segformer(pixel_values=pixel_values).logits 
        b, _, h, w = seg_logits.shape
        hints_spatial = hints.view(b, 11, 1, 1).expand(-1, -1, h, w)
        fused_features = torch.cat([seg_logits, hints_spatial], dim=1)
        final_logits = self.fusion_conv(fused_features)
        return F.interpolate(final_logits, size=pixel_values.shape[-2:], mode="bilinear", align_corners=False)

model = FusionSegFormer(num_classes=11)
model.load_state_dict(torch.load('fusion_segformer_epoch_2.pth', weights_only=True))
model = model.to(device)
model.eval()

Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b0 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


FusionSegFormer(
  (segformer): SegformerForSemanticSegmentation(
    (segformer): SegformerModel(
      (encoder): SegformerEncoder(
        (patch_embeddings): ModuleList(
          (0): SegformerOverlapPatchEmbeddings(
            (proj): Conv2d(3, 32, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
            (layer_norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
          )
          (1): SegformerOverlapPatchEmbeddings(
            (proj): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
            (layer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          )
          (2): SegformerOverlapPatchEmbeddings(
            (proj): Conv2d(64, 160, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
            (layer_norm): LayerNorm((160,), eps=1e-05, elementwise_affine=True)
          )
          (3): SegformerOverlapPatchEmbeddings(
            (proj): Conv2d(160, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
            (l

In [3]:
num_classes = 11
total_inter = torch.zeros(num_classes).to(device)
total_union = torch.zeros(num_classes).to(device)
save_dir = r'D:\Study\HumanStudy\Dataset\Validation_Predictions_2'

os.makedirs(save_dir, exist_ok=True)
model.eval()
classifier.eval()

class_names = ["배경(0)", "균열(1)", "누수(2)", "백태(3)", "박리(4)", "망상균열(5)", 
               "박락(6)", "재료분리(7)", "철근노출(8)", "파손(9)", "들뜸(10)"]

with torch.no_grad():
    for images, masks, img_paths in tqdm(val_loader):
        images, masks = images.to(device), masks.to(device)
        
        images_cls = F.interpolate(images, size=(224, 224), mode='bilinear', align_corners=False)
        hints = torch.sigmoid(classifier(images_cls))
        
        logits = model(images, hints)
        probs = torch.sigmoid(logits)
        
        preds = (probs > 0.4).int()
        
        masks_one_hot = F.one_hot(masks, num_classes=11).permute(0, 3, 1, 2).int()
        intersection = (preds & masks_one_hot).float().sum(dim=(0, 2, 3))
        union = (preds | masks_one_hot).float().sum(dim=(0, 2, 3))
        total_inter += intersection
        total_union += union
        
        max_probs, max_indices = torch.max(probs, dim=1)
        batch_pred_masks = torch.where(max_probs > 0.5, max_indices, torch.zeros_like(max_indices))
        
        for i in range(images.size(0)):
            raw_mask = batch_pred_masks[i].cpu().numpy().astype(np.uint8)
            original_stem = Path(img_paths[i]).stem
            save_path = os.path.join(save_dir, f"{original_stem}.png")
            
            is_success, encoded_img = cv2.imencode('.png', raw_mask)
            if is_success:
                encoded_img.tofile(save_path)

ious = total_inter / (total_union + 1e-6)
miou = ious[1:].mean().item()

print(f"Mean IoU: {miou * 100:.2f}%\n")
for i in range(11):
    print(f" - {class_names[i]:<10}: {ious[i].item() * 100:.2f}%")

100%|████████████████████████████████████████████████████████████████████████████| 13125/13125 [58:07<00:00,  3.76it/s]

Mean IoU: 11.39%

 - 배경(0)     : 98.38%
 - 균열(1)     : 20.60%
 - 누수(2)     : 38.83%
 - 백태(3)     : 23.66%
 - 박리(4)     : 5.54%
 - 망상균열(5)   : 2.18%
 - 박락(6)     : 4.20%
 - 재료분리(7)   : 14.50%
 - 철근노출(8)   : 0.91%
 - 파손(9)     : 3.32%
 - 들뜸(10)    : 0.16%
